In [31]:
import pandas as pd
import psycopg
import mplfinance as mpf
from pathlib import Path
import json
import zipfile


In [43]:
START_DATE = "2026-01-05"
END_DATE = "2026-04-24"
BATCH_SIZE = 5
CUTOFF_TIME = "08:22:00"

In [44]:
conn = psycopg.connect(
    "dbname=dailyedge_development"
)

In [45]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT timestamp::date
        FROM candles
        WHERE timestamp::date BETWEEN %s AND %s
          AND timestamp::time = '08:30:00'
        ORDER BY timestamp::date;
    """, (START_DATE, END_DATE))

    trading_dates = [row[0] for row in cur.fetchall()]

In [46]:
batches = [
    trading_dates[i:i + BATCH_SIZE]
    for i in range(0, len(trading_dates), BATCH_SIZE)
]

[len(batch) for batch in batches], batches

([5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4],
 [[datetime.date(2026, 1, 5),
   datetime.date(2026, 1, 6),
   datetime.date(2026, 1, 7),
   datetime.date(2026, 1, 8),
   datetime.date(2026, 1, 9)],
  [datetime.date(2026, 1, 12),
   datetime.date(2026, 1, 13),
   datetime.date(2026, 1, 14),
   datetime.date(2026, 1, 15),
   datetime.date(2026, 1, 16)],
  [datetime.date(2026, 1, 19),
   datetime.date(2026, 1, 20),
   datetime.date(2026, 1, 21),
   datetime.date(2026, 1, 22),
   datetime.date(2026, 1, 23)],
  [datetime.date(2026, 1, 26),
   datetime.date(2026, 1, 27),
   datetime.date(2026, 1, 28),
   datetime.date(2026, 1, 29),
   datetime.date(2026, 1, 30)],
  [datetime.date(2026, 2, 2),
   datetime.date(2026, 2, 3),
   datetime.date(2026, 2, 4),
   datetime.date(2026, 2, 5),
   datetime.date(2026, 2, 6)],
  [datetime.date(2026, 2, 9),
   datetime.date(2026, 2, 10),
   datetime.date(2026, 2, 11),
   datetime.date(2026, 2, 12),
   datetime.date(2026, 2, 13)],
  [datetime.date(2026, 2

In [47]:
def build_replay(target_date):
    target_date = pd.Timestamp(target_date)

    replay_dir = Path("replays") / str(target_date.date())
    replay_dir.mkdir(parents=True, exist_ok=True)

    start_time = target_date - pd.Timedelta(days=565)
    cutoff = pd.Timestamp(f"{target_date.date()} {CUTOFF_TIME}")

    query = """
        SELECT timestamp, open, high, low, close, volume
        FROM CANDLES
        WHERE timestamp >= %s
        AND timestamp <= %s
        ORDER BY timestamp
    """

    df = pd.read_sql(query, conn, params=(start_time, cutoff))
    df = df.set_index("timestamp")

    df_5m = df.resample("5min").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    }).dropna()

    df_30m = df.resample("30min").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    }).dropna()

    df_4h = df.resample("4h", offset="1h").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    }).dropna()

    df_1d = df.resample("24h", offset="17h").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    }).dropna()

    current_price = df.iloc[-1]["close"]

    overnight_start = target_date - pd.Timedelta(days=1) + pd.Timedelta(hours=17)

    overnight = df.loc[overnight_start:cutoff]

    overnight_high = overnight["high"].max()
    overnight_low = overnight["low"].min()
    overnight_range = overnight_high - overnight_low

    previous_date_query = """
        SELECT MAX(timestamp::date)
        FROM candles
        WHERE timestamp::date < %s
        AND timestamp::time = '15:15:00'
    """

    previous_date = pd.read_sql(
        previous_date_query,
        conn,
        params=(target_date.date(),)
    ).iloc[0, 0]

    previous_date = pd.Timestamp(previous_date)

    previous_rth = df.loc[
        f"{previous_date.date()} 08:30:00":
        f"{previous_date.date()} 15:00:00"
    ]

    previous_day_range = previous_rth["high"].max() - previous_rth["low"].min()

    previous_close_time = pd.Timestamp(
        f"{previous_date.date()} 15:15:00"
    )

    previous_close = df.loc[previous_close_time, "close"]

    org = current_price - previous_close

    completed_daily = df_1d.iloc[:-1].copy()

    completed_daily["prev_close"] = completed_daily["close"].shift(1)

    completed_daily["true_range"] = pd.concat([
        completed_daily["high"] - completed_daily["low"],
        (completed_daily["high"] - completed_daily["prev_close"]).abs(),
        (completed_daily["low"] - completed_daily["prev_close"]).abs()
    ], axis=1).max(axis=1)

    atr_14 = completed_daily["true_range"].tail(14).mean()

    context = {
        "date": str(target_date.date()),
        "cutoff_time_ct": CUTOFF_TIME,
        "current_price": round(current_price, 2),
        "atr_14": round(atr_14, 2),
        "overnight_high": round(overnight_high, 2),
        "overnight_low": round(overnight_low, 2),
        "overnight_range": round(overnight_range, 2),
        "previous_day_range": round(previous_day_range, 2),
        "org": round(org, 2),
    }

    chart_1m = df.loc[
        f"{target_date.date()} 02:00:00":
        cutoff
    ]

    chart_5m = df_5m.loc[
        f"{previous_date.date()} 05:30:00":
        f"{target_date.date()} {CUTOFF_TIME}"
    ]

    chart_30m = df_30m.loc[
        target_date - pd.Timedelta(days=12):
        cutoff
    ]

    chart_4h = df_4h.loc[target_date - pd.Timedelta(days=90):cutoff]

    chart_1d = df_1d.loc[start_time:cutoff]

    return {
        "target_date": target_date,
        "replay_dir": replay_dir,
        "context": context,
        "df": df,
        "chart_1m": chart_1m,
        "chart_5m": chart_5m,
        "chart_30m": chart_30m,
        "chart_4h": chart_4h,
        "chart_1d": chart_1d,
        "df_5m": df_5m,
    }


In [37]:
def save_charts(replay):
    target_date = replay["target_date"]
    replay_dir = replay["replay_dir"]
    chart_1m = replay["chart_1m"]
    chart_5m = replay["chart_5m"]
    chart_30m = replay["chart_30m"]
    chart_4h = replay["chart_4h"]
    chart_1d = replay["chart_1d"]

    mpf.plot(
        chart_1m,
        type="candle",
        volume=True,
        figsize=(22, 8),
        title=f"MNQ 1m — {target_date.date()} — cutoff {CUTOFF_TIME} CT",
        warn_too_much_data=1000,
        savefig=replay_dir / "1m.png"
    )
    mpf.plot(
        chart_5m,
        type="candle",
        volume=True,
        figsize=(22, 8),
        title=f"MNQ 5m — {target_date.date()} — cutoff {CUTOFF_TIME} CT",
        warn_too_much_data=1000,
        savefig=replay_dir / "5m.png"
    )
    mpf.plot(
        chart_30m,
        type="candle",
        volume=True,
        figsize=(22, 8),
        title=f"MNQ 30m — {target_date.date()} — cutoff {CUTOFF_TIME} CT",
        warn_too_much_data=1000,
        savefig=replay_dir / "30m.png"
    )
    mpf.plot(
        chart_4h,
        type="candle",
        volume=True,
        figsize=(22, 8),
        title=f"MNQ 4h — {target_date.date()} — cutoff {CUTOFF_TIME} CT",
        warn_too_much_data=1000,
        savefig=replay_dir / "4h.png"
    )
    mpf.plot(
        chart_1d,
        type="candle",
        volume=True,
        figsize=(22, 8),
        title=f"MNQ 1d — {target_date.date()} — cutoff {CUTOFF_TIME} CT",
        warn_too_much_data=1000,
        savefig=replay_dir / "1D.png"
    )

In [38]:
def save_replay_data(replay):
    replay_dir = replay["replay_dir"]
    context = replay["context"]
    df_5m = replay["df_5m"]

    with open(replay_dir / "context.json", "w") as f:
        json.dump(
            context,
            f,
            indent=2,
            default=lambda x: x.item()
        )

    df_5m.tail(200).to_csv(replay_dir / "nq_data.csv")


In [39]:
study_dates = pd.read_sql(
    """
    SELECT DISTINCT DATE(timestamp) AS trading_date
    FROM CANDLES
    WHERE timestamp >= %s
      AND timestamp < %s
      AND EXTRACT(ISODOW FROM timestamp) BETWEEN 1 AND 5
    ORDER BY trading_date
    """,
    conn,
    params=(START_DATE, END_DATE),
)["trading_date"].astype(str).tolist()

len(study_dates), study_dates[:3], study_dates[-3:]

/tmp/ipykernel_112313/3601155803.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  study_dates = pd.read_sql(


(0, [], [])

In [40]:
for target_date in study_dates:
    # print(f"Generating {target_date}...")

    replay = build_replay(target_date)
    save_charts(replay)
    save_replay_data(replay)

In [41]:
# Not invloved for batch-generation. For inspection only

def load_prediction_replay(target_date):
    replay_dir = Path("replays") / str(target_date)

    with open(replay_dir / "context.json", "r") as f:
        context = json.load(f)

    df_5m = pd.read_csv(replay_dir / "nq_data.csv")

    chart_paths = {
        "1m": replay_dir / "1m.png",
        "5m": replay_dir / "5m.png",
        "30m": replay_dir / "30m.png",
        "4h": replay_dir / "4h.png",
        "1D": replay_dir / "1D.png",
    }

    return {
        "date": str(target_date),
        "context": context,
        "df_5m": df_5m,
        "charts": chart_paths,
    }

In [48]:
REPLAY_DIR = Path("replays")
ZIP_DIR = Path("replay_batches")
BATCH_SIZE = 5

ZIP_DIR.mkdir(exist_ok=True)

replay_dirs = sorted(
    (
        path for path in REPLAY_DIR.iterdir()
        if path.is_dir()
        and START_DATE <= path.name <= END_DATE
    ),
    reverse=True
)

for batch_num, start in enumerate(range(0, len(replay_dirs), BATCH_SIZE), start=0):
    batch_num = -batch_num
    batch = replay_dirs[start:start + BATCH_SIZE]

    zip_path = ZIP_DIR / f"replay_batch_{batch_num}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for replay_dir in batch:
            for file_path in replay_dir.iterdir():
                if file_path.is_file():
                    arcname = Path(replay_dir.name) / file_path.name
                    zf.write(file_path, arcname)

    print(
        f"{zip_path} — "
        f"{batch[0].name} → {batch[-1].name} "
        f"({len(batch)} sessions)"
    )

replay_batches/replay_batch_0.zip — 2026-04-23 → 2026-04-17 (5 sessions)
replay_batches/replay_batch_-1.zip — 2026-04-16 → 2026-04-10 (5 sessions)
replay_batches/replay_batch_-2.zip — 2026-04-09 → 2026-04-03 (5 sessions)
replay_batches/replay_batch_-3.zip — 2026-04-02 → 2026-03-27 (5 sessions)
replay_batches/replay_batch_-4.zip — 2026-03-26 → 2026-03-20 (5 sessions)
replay_batches/replay_batch_-5.zip — 2026-03-19 → 2026-03-13 (5 sessions)
replay_batches/replay_batch_-6.zip — 2026-03-12 → 2026-03-06 (5 sessions)
replay_batches/replay_batch_-7.zip — 2026-03-05 → 2026-02-27 (5 sessions)
replay_batches/replay_batch_-8.zip — 2026-02-26 → 2026-02-20 (5 sessions)
replay_batches/replay_batch_-9.zip — 2026-02-19 → 2026-02-13 (5 sessions)
replay_batches/replay_batch_-10.zip — 2026-02-12 → 2026-02-06 (5 sessions)
replay_batches/replay_batch_-11.zip — 2026-02-05 → 2026-01-30 (5 sessions)
replay_batches/replay_batch_-12.zip — 2026-01-29 → 2026-01-23 (5 sessions)
replay_batches/replay_batch_-13.zip 